# Sprint 2 — Working with Text Data

Leitura integral do Capítulo 2 – Working with Text Data, conforme a referência principal do projeto

Durante a leitura, procure compreender:

    1- como um texto é dividido em unidades menores;
    2- o que é um token;
    3- como os tokens são associados a identificadores numéricos;
    4- como é construído um vocabulário;
    5- como uma sequência de texto é transformada em uma sequência de números;
    6- como são formados os pares de entrada e alvo utilizados no treinamento;
    7- por que os tokens precisam ser convertidos em vetores;
    8- o papel dos embeddings;
    9- a necessidade de representar a posição dos tokens;
    10- como os dados são organizados para serem fornecidos ao modelo.

## Respostas

1- Através de um processo chamado tokenização, em que o texto é dividido em unidades menores chamadas tokens: palavras, partes de palavras e sinais de pontuação. No tokenizador simples do capítulo isso é feito com uma expressão regular que separa espaços e pontuação; no BPE (usado no GPT) palavras raras são quebradas em pedaços menores (subpalavras).

2- Token é a menor unidade de texto que o modelo processa. Pode ser uma palavra ("the"), parte de uma palavra ("terr" + "aces") ou um símbolo ("," "?"). Cada token depois é trocado por um número (Token ID) e, em seguida, por um vetor.

3- Através de um vocabulário: um dicionário que associa cada token distinto a um número inteiro único, o Token ID. O vocabulário também permite o caminho inverso (ID → token), usado no decode.

4- O vocabulário é construído como se fosse um dicionário: o texto de entrada é tokenizado e quebrado em tokens individuais, os tokens duplicados são removidos, os que sobram são ordenados em ordem alfabética e cada um recebe um ID único. No final acrescentamos os tokens especiais `<|endoftext|>` (separa textos diferentes) e `<|unk|>` (representa palavras que não estão no vocabulário). Com o the-verdict.txt o vocabulário tem 1.130 tokens, ou 1.132 com os especiais.

5- A tokenização quebra o texto em tokens, cada token é procurado no vocabulário e trocado pelo seu Token ID. O resultado é uma lista de números inteiros. Exemplo com o BPE do GPT-2: "Hello, do you like tea?" → [15496, 11, 466, 345, 588, 8887, 30].

6- Com uma janela deslizante: a entrada x é um trecho de `max_length` tokens e o alvo y é o mesmo trecho deslocado uma posição para a direita. Para cada posição de x o modelo deve prever o token seguinte, que está na mesma posição de y. Exemplo: x = [290, 4920, 2241, 287] e y = [4920, 2241, 287, 257] — "and" → "established", "and established" → "himself", e assim por diante.

7- Porque o modelo não tem como traduzir e entender texto cru: a rede neural só faz operações matemáticas (multiplicações de matrizes), então o texto precisa virar números. E os Token IDs sozinhos também não bastam, porque são só índices — o ID 500 não é "parecido" com o 501. Por isso cada ID é trocado por um vetor de números reais, onde a semelhança pode ser medida.

8- O embedding transforma cada Token ID em um vetor (por exemplo, de 256 dimensões). Na prática é uma tabela (`nn.Embedding`) com uma linha por token do vocabulário: o ID escolhe a linha. Os valores começam aleatórios e são aprendidos no treinamento, de forma que tokens usados de forma parecida fiquem com vetores parecidos.

9- O embedding de token é o mesmo em qualquer posição: "the" no começo e no fim da frase recebe exatamente o mesmo vetor. E o mecanismo de atenção (Sprint 3) também não percebe a ordem. Sem informação de posição, "the dog bit the man" e "the man bit the dog" teriam a mesma representação. Por isso somamos ao embedding de cada token um embedding posicional, um vetor diferente para cada posição.

10- Em lotes: o `Dataset` recorta os pares (x, y) com a janela deslizante e o `DataLoader` agrupa esses pares em tensores de forma (batch_size, max_length). Esses IDs passam pelo embedding de token, recebem o embedding posicional e viram o tensor (batch_size, max_length, d_emb), que é a entrada do modelo.

> O arquivo `the-verdict.txt` está nesta mesma pasta (`sprints/`), então o notebook não precisa baixá-lo. Execute-o a partir de `sprints/`.

## 1. Tokenização (seção 2.2)

In [1]:
#Teste do Livro — leitura do corpus

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [2]:
import re  #biblioteca para separar palavras
text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text) #função que faz a separação das palavras
print(result)

#é possivel notar que ele divide inclusive os espaços, pontos e vigulas do texto o que posteriormente vamos tirar e deixar mais limpo

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [3]:
import re  #biblioteca para separar palavras
text = "Hello, world. This, is a test."
result = re.split(r'([,.]|\s)', text) #função que faz a separação das palavras e também dos pontos, virgulas e espaços
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [4]:
import re  #biblioteca para separar palavras
text = "Hello, world. This, is a test."
result = [item for item in result if item.strip()] #função que faz a separação das palavras e também dos pontos, virgulas e remove redundancias
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [5]:
import re  #biblioteca para separar palavras
text = "Hello, world. This, is a test."
result = re.split(r'([,.]|\s)', text)
result = [item for item in result if item.strip()] #função que faz a separação das palavras e também dos pontos, virgulas e remove redundancias
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [6]:
import re
#Sequencia de comandos para transformar o texto cru do livro em tokens, separar palavras, remover redundancias, aparesentar quantiade
#de tokens e apresentar as tokens do texto


preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed)) # quantidade de tokens do texto inteiro

print(preprocessed[:30]) #primeiras 30 tokens do texto tem como alterar o valor

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## 2. Vocabulário e Token IDs (seções 2.3 e 2.4)

In [7]:
#codigo para determinar o tamanho de um vocabulario



preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size) # mostra a quantidade de palavras/simbolos unicos no texto


1130


In [8]:
#codigo para criar um vocabulario



preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

vocab = {token:integer for integer,token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break


('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [9]:
#codigo para criar uma classe de tokenização simples

import re


with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

PADRAO = r'([,.:;?_!"()\']|--|\s)'      # um só, usado nos dois lugares

def tokenize(texto):
    pecas = re.split(PADRAO, texto)
    return [p.strip() for p in pecas if p.strip()]

preprocessed = tokenize(raw_text)

all_words = sorted(set(preprocessed))
vocab = {token: integer for integer, token in enumerate(all_words)}
print("vocab_size:", len(vocab))


class SimpleTokenizerV1: #aqui ele vai rodar um vocabulario proprio e codificar os tokens e por fim o decode vai transformar em texto denovo
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = tokenize(text)
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        return re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)   # tira espaço antes da pontuação


tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," 
       Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids) #printa os tokens do texto de exemplo codificados já como tokens IDs

print(tokenizer.decode(ids)) #printa o texto depois de decodificar

# Adição de tokens especiais ao vocabulário para integrar palavras desconhecidas
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}
print(len(vocab.items()))

vocab_size: 1130
[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.
1132


In [10]:
#codigo para teste de texto com tokenização e capacidade para reconhecer palavras desconhecidas que não existem no texto de exemplo

import re


with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

PADRAO = r'([,.:;?_!"()\']|--|\s)'

def tokenize(texto):
    pecas = re.split(PADRAO, texto)
    return [p.strip() for p in pecas if p.strip()]

preprocessed = tokenize(raw_text)

# vocabulário JÁ com os tokens especiais, ANTES de criar o tokenizer
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: integer for integer, token in enumerate(all_tokens)}
print("vocab_size:", len(vocab))


class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = tokenize(text)
        preprocessed = [item if item in self.str_to_int else "<|unk|>"
                        for item in preprocessed]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        return re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)


tokenizer = SimpleTokenizerV2(vocab)

text = """"It's the last he painted, you know,"
       Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

print(tokenizer.decode(tokenizer.encode(text)))

vocab_size: 1132
[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.
Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## 3. Byte Pair Encoding com tiktoken (seção 2.5)

O tokenizador simples só conhece as palavras do conto: tudo o que não está no vocabulário vira `<|unk|>`. O BPE do GPT-2 tem 50.257 tokens e quebra palavras desconhecidas em subpalavras, então nunca precisa de `<|unk|>` e o `decode` reconstrói o texto exatamente.

In [11]:
#Codigo para utilizar o TikToken, o tokenizador Byte Pair Encoding (BPE) usado no GPT-2
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces"
        " of someunknownPlace.")
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)
print(tokenizer.decode(integers))

#palavra que nao existe: o BPE quebra em pedacos menores em vez de usar <|unk|>
desconhecida = tokenizer.encode("Akwirw ier")
print(desconhecida, [tokenizer.decode([i]) for i in desconhecida])

print("tamanho do vocabulario:", tokenizer.n_vocab)
print("tokens BPE no conto:", len(tokenizer.encode(raw_text)))

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.
[33901, 86, 343, 86, 220, 959] ['Ak', 'w', 'ir', 'w', ' ', 'ier']
tamanho do vocabulario: 50257
tokens BPE no conto: 5145


## 4. Pares de entrada e alvo (seção 2.6)

In [12]:
#Codigo para criar os pares de entrada (x) e alvo (y) com o tokenizador simples:
#o alvo e a entrada deslocada uma posicao — o modelo aprende a prever o proximo token
import re


with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

PADRAO = r'([,.:;?_!"()\']|--|\s)'

def tokenize(texto):
    pecas = re.split(PADRAO, texto)
    return [p.strip() for p in pecas if p.strip()]

preprocessed = tokenize(raw_text)

# vocabulário JÁ com os tokens especiais, ANTES de criar o tokenizer
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: integer for integer, token in enumerate(all_tokens)}
print("vocab_size:", len(vocab))


class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = tokenize(text)
        preprocessed = [item if item in self.str_to_int else "<|unk|>"
                        for item in preprocessed]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        return re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)


tokenizer = SimpleTokenizerV2(vocab)

text = """"It's the last he painted, you know,"
       Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)
#print(ids)
#print(tokenizer.decode(ids))

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
#print(text)

tokenizer = SimpleTokenizerV2(vocab)
#print(tokenizer.encode(text))

#print(tokenizer.decode(tokenizer.encode(text)))

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

enc_sample = enc_text[50:]


context_size = 4        
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))


vocab_size: 1132
4690
x: [568, 115, 1066, 727]
y:      [115, 1066, 727, 988]
[568] ----> 115
[568, 115] ----> 1066
[568, 115, 1066] ----> 727
[568, 115, 1066, 727] ----> 988
in ----> a
in a ----> villa
in a villa ----> on
in a villa on ----> the


In [13]:
#Os mesmos pares com o BPE (valores do livro)
#a celula anterior trocou tokenizer pelo tokenizador simples, entao voltamos ao BPE
tokenizer = tiktoken.get_encoding("gpt2")

enc_text = tokenizer.encode(raw_text)
enc_sample = enc_text[50:]

context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]
 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


## 5. Dataset e DataLoader (seção 2.6)

In [14]:
#codigo de criação de um dataset para os inputs e pares criados anteriormente
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)

        #janela deslizante: anda de stride em stride e recorta max_length tokens
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)


#lote de 1 com stride 1: cada lote anda 1 token
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
print(next(data_iter))
print(next(data_iter))

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [15]:
#lote de 8 com stride 4: janelas encostadas, sem sobreposicao
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
inputs, targets = next(iter(dataloader))
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)
print("\namostras no dataset:", len(dataloader.dataset), "| lotes por epoca:", len(dataloader))

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])

amostras no dataset: 1286 | lotes por epoca: 160


## 6. Embeddings de token (seção 2.7)

O `nn.Embedding` é uma tabela de pesos com uma linha por token: o Token ID escolhe a linha.

In [16]:
#exemplo pequeno do livro: vocabulario de 6 tokens e vetores de 3 dimensoes
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(6, 3)
print(embedding_layer.weight)

#o vetor do token 3 e a linha 3 da tabela
print(embedding_layer(torch.tensor([3])))
print(embedding_layer(torch.tensor([2, 3, 5, 1])))

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [17]:
#tamanho real: vocabulario do GPT-2 (50257) e vetores de 256 dimensoes
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

token_embeddings = token_embedding_layer(inputs)   #inputs do lote (8, 4)
print("token IDs       :", inputs.shape)
print("token embeddings:", token_embeddings.shape)   #(8, 4, 256)

token IDs       : torch.Size([8, 4])
token embeddings: torch.Size([8, 4, 256])


## 7. Embeddings posicionais (seção 2.8)

O embedding de token não depende da posição: o mesmo token recebe o mesmo vetor em qualquer lugar da frase. Somamos um segundo embedding, com um vetor por posição.

In [18]:
context_length = 4   #max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print("pos embeddings  :", pos_embeddings.shape)     #(4, 256)

#o broadcasting soma o mesmo vetor de posicao em todas as 8 sequencias do lote
input_embeddings = token_embeddings + pos_embeddings
print("input embeddings:", input_embeddings.shape)   #(8, 4, 256) -> entrada do modelo

pos embeddings  : torch.Size([4, 256])
input embeddings: torch.Size([8, 4, 256])


In [19]:
#demonstracao: o token ' the' repetido 3 vezes
ids = torch.tensor(tokenizer.encode(" the the the"))
print(ids)

sem_posicao = token_embedding_layer(ids)
com_posicao = token_embedding_layer(ids) + pos_embedding_layer(torch.arange(len(ids)))

cos = torch.nn.functional.cosine_similarity
print("sem posicao, 1a x 3a ocorrencia:", cos(sem_posicao[0], sem_posicao[2], dim=0).item())
print("com posicao, 1a x 3a ocorrencia:", cos(com_posicao[0], com_posicao[2], dim=0).item())

tensor([262, 262, 262])
sem posicao, 1a x 3a ocorrencia: 1.0
com posicao, 1a x 3a ocorrencia: 0.5293339490890503


## 8. Integração com o código do projeto (`src/`)

A versão organizada deste pipeline está em `src/` — `tokenizacao.py`, `dados.py` e `embeddings.py` — e é a que as próximas sprints importam. A célula abaixo refaz o mesmo lote usando esses módulos e confere que o resultado é igual ao deste notebook. A validação completa, contra os valores do livro, está em `src/validar_dados.py`.

In [20]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from dados import carregar_texto, criar_dataloader
from embeddings import EmbeddingDeEntrada
from tokenizacao import TokenizadorBPE, TokenizadorSimples

simples = TokenizadorSimples.a_partir_do_texto(carregar_texto())
print("vocabulario simples:", simples.tamanho_vocabulario)
print(simples.encode("Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace."))

lote, alvos = next(iter(criar_dataloader(carregar_texto(), batch_size=8, max_length=4, stride=4)))
print("mesmo lote do notebook:", torch.equal(lote, inputs), torch.equal(alvos, targets))

camada = EmbeddingDeEntrada(vocab_size=50257, d_emb=256, context_length=4)
print("entrada do modelo:", camada(lote).shape)

vocabulario simples: 1132
[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
mesmo lote do notebook: True True
entrada do modelo: torch.Size([8, 4, 256])


## Próximos passos

- Os experimentos da sprint estão em `experiments/sprint02-texto/` e os resultados em `results-by-sprints/sprint02/`.
- A análise, com as respostas às perguntas da sprint apoiadas nos experimentos, está em `reports/sprint02-relatorio.md`.
- O glossário do capítulo está em `technical-glossary/capitulo-02.md`.
- O tensor `(batch_size, max_length, d_emb)` produzido aqui é a entrada dos mecanismos de atenção da Sprint 3.